## 1. Conexão com o Drive e instalação de dependências
Monta o Drive para acessar `dados/` e salvar artefatos e `resultados/`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1.2 — Dependências e imports

In [ ]:
!pip install -q --upgrade "torchao>=0.16.0"
!pip install -q scikit-learn peft transformers accelerate openpyxl xlsxwriter ftfy
import os, gc, json, time, random, html, re, unicodedata, ftfy
from datetime import datetime
from pathlib import Path
from collections import Counter
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.model_selection import StratifiedGroupKFold,StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    f1_score, accuracy_score, matthews_corrcoef,
    classification_report, confusion_matrix,
)
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, TaskType
import hashlib
import torch.nn.functional as F
from collections import Counter, defaultdict

## 1.3 — Verificação de hardware

In [ ]:
print("CUDA:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("Torch:", torch.__version__)

CUDA: True
Device: NVIDIA RTX PRO 6000 Blackwell Server Edition
Torch: 2.11.0+cu130


## 2 — Configuração global

Mudanças em relação ao v8(11):
- **`SEEDS_ENSEMBLE = [42, 123, 2024]`**: reduz variância do fold. O `best_epoch`
  oscilava entre 2 e 5, sinal de instabilidade — 3 seeds por fold dão uma
  estimativa muito mais estável.
- **`FOLDS_TO_RUN = [0, 1, 2]`**: com 1 fold, 1 ponto de acurácia ≈ ruído.
  Precisamos do desvio-padrão entre folds para decidir se uma variante
  realmente vence.
- **`VARIANTES = ["A", "B", "D", "E"]`**: C removida (dominada por A, mesmo
  agrupamento com alvo pior). E é nova: texto **cru** + soft labels, para
  isolar o efeito do pré-processamento.
- **`epochs=6`, `scheduler="cosine"`**: o `best_epoch` variava demais. Cosine
  estabiliza o fim do treino e costuma dar +0.5–1 pt.
- **`lr_clf=5e-4`**: o grid do V6 apontou 5e-4 como melhor para o classificador.
- **`focal_gamma=2.0`**: `c234` sempre tem F1 ~0.4; focal loss foca nos
  exemplos difíceis e costuma ajudar classes confusas.


In [ ]:
SEED        = 42
SEEDS_ENSEMBLE = [42, 123, 2024]
N_FOLDS     = 10
FOLDS_TO_RUN = [0, 1, 2]
MIN_CHARS   = 30
CLASSES     = ["c1", "c234", "c5"]
class_to_idx = {c: i for i, c in enumerate(CLASSES)}
WARMUP_PCT  = 0.10
MAX_LENGTH  = 384

CFG_TREINO = dict(
    model="neuralmind/bert-base-portuguese-cased",
    batch=16, grad_accum=1,
    epochs=6,
    max_length=384,
    lora_r=32, lora_alpha=64, lora_dropout=0.1,
    target_modules=["query", "key", "value", "output.dense", "intermediate.dense"],
    lr_lora=3e-5,
    lr_clf=5e-4,
    label_smoothing=0.05,
    focal_gamma=2.0,
    scheduler="cosine",
)

VARIANTES = ["A", "B", "D", "E"]
DRIVE_ROOT = Path("/content/drive/MyDrive/EP_NLP")
TRAIN_PATH = DRIVE_ROOT / "dados" / "train.xlsx"
TEST_PATH  = DRIVE_ROOT / "dados" / "test1.xlsx"

(DRIVE_ROOT / "resultados" / "v9_final").mkdir(parents=True, exist_ok=True)

## 3 — Utilitários

- `set_seed`: fixa todas as seeds (Python, NumPy, PyTorch, CUDA) para reprodutibilidade.
- `Logger`: escreve no stdout e em arquivo simultaneamente, para preservar histórico entre sessões do Colab.
- `normalizar_grupo`: chave de agrupamento para `StratifiedGroupKFold`. Usa apenas os primeiros 200 caracteres normalizados — captura duplicatas e near-duplicates que começam igual.
- `hash_grupo`: variante estrita — hash do texto inteiro normalizado. Só agrupa duplicatas exatas.
- `extract/load_trainable_state`: serializa apenas LoRA + classifier, deixando o backbone BERT intacto (economiza ~99% do espaço).

In [ ]:
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

class Logger:
    def __init__(self, path):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self._f = open(self.path, "a", encoding="utf-8")
    def __call__(self, msg=""):
        print(msg)
        self._f.write(str(msg) + "\n")
        self._f.flush()
    def close(self):
        self._f.close()

def normalizar_grupo(s: str, prefix: int = 200) -> str:
    """Chave de grupo para StratifiedGroupKFold.
    Deve ser estável e capturar duplicatas exatas/quase-exatas."""
    if not isinstance(s, str):
        return ""
    s = s.lower()
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("utf-8")
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^a-z0-9\s]", "", s).strip()
    return s[:prefix]

def hash_grupo(s: str) -> str:
    if not isinstance(s, str): return ""
    s = s.lower()
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("utf-8")
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^a-z0-9\s]", "", s).strip()
    return hashlib.md5(s.encode()).hexdigest()

## 4 — Pré-processamento de texto

Sem mudança. O pipeline continua: encoding → saudações/rodapé → mascarar
entidades → normalizar espaços.

**Por que E existe:** todo o pipeline remove sinais que podem ser úteis
(comprimento, estrutura, formatação, nomes de órgãos). A variante E treina
com `resp_text` cru para medir se o pré-processamento está ajudando ou
atrapalhando. É uma ablação barata e informativa.

In [ ]:
SAUDACAO_RE = re.compile(
    r'^\s*(?:'
    r'Prezad[oa](?:\s*\(a\))?'
    r'(?:\s+(?:Senhor(?:a)?|Sr\.?|Sra\.?|Srs\.?|Senhores|Cidadã[o]?|Caro\(a\)|Caro|Cara))?'
    r'|Caro(?:s)?(?:\s+(?:Senhor(?:es)?|Sr\.?|Sra\.?|Colega))?'
    r'|Senhor(?:a|es)?'
    r'|Sr\.?|Sra\.?'
    r'|Ilmo\.?\s*|Ilma\.?\s*'
    r'|Excelent[íi]ssim[oa]\s+Senhor(?:a)?'
    r'|Dign[íi]ssim[oa]'
    r'|Ol[áa]'
    r')'
    r'[^\n]{0,120}?[,\!\.:]\s*',
    re.IGNORECASE,
)
BOM_DIA_RE = re.compile(r'^\s*(?:Bom dia|Boa tarde|Boa noite)[!.,\s]*', re.IGNORECASE)
FECHO_RE = re.compile(
    r'(?i)(?:^|[\s\.\,\!\?\;\:\)\]])\s*'
    r'(?:Atenciosamente|Cordialmente|Respeitosamente|Sinceramente'
    r'|Abra[çc]os?|Att\.?|At\.te'
    r'|Sem\s+mais\s+para\s+o\s+momento)\b'
)

def corrigir_encoding(t):
    if not isinstance(t, str):
        return ""
    t = ftfy.fix_encoding(t)
    t = html.unescape(t)
    t = re.sub(r"&deg,?",   "º",  t)
    t = re.sub(r"&bull,?",  "•",  t)
    t = re.sub(r"&hellip,?", "...", t)
    t = re.sub(r"&[a-zA-Z]+;", " ", t)
    t = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", " ", t)
    t = re.sub(r"[\u0091-\u0094\u0096\u0097]", "", t)
    t = t.replace("\u00a0", " ")
    t = re.sub(r"[\u200b-\u200d\ufeff]", "", t)
    return t

def mascarar_entidades(t):
    t = re.sub(r"https?://\S+|www\.\S+", " [URL] ", t)
    t = re.sub(r"\S+@\S+", " [EMAIL] ", t)
    t = re.sub(r"\(?\d{2}\)?\s?\d{4,5}[-\s]?\d{4}", " [TELEFONE] ", t)
    t = re.sub(r"\d{5}\.\d{6}/\d{4}-\d{2}", " [NUP] ", t)
    t = re.sub(r"\d{2}/\d{2}/\d{4}", " [DATA] ", t)
    t = re.sub(r"\d{3}\.\d{3}\.\d{3}-\d{2}", " [CPF] ", t)
    t = re.sub(r"\d{2}\.\d{3}\.\d{3}/\d{4}-\d{2}", " [CNPJ] ", t)
    t = re.sub(r"\b\d{6,}\b", " [NUM] ", t)
    return t

def limpar_saudacao_rodape(t):
    if not isinstance(t, str) or not t:
        return t
    t = SAUDACAO_RE.sub('', t, count=1)
    t = BOM_DIA_RE.sub('', t, count=1)
    candidatos = list(FECHO_RE.finditer(t))
    if candidatos:
        limite = max(int(0.35 * len(t)), len(t) - 2500)
        finais = [m for m in candidatos if m.start() >= limite]
        if finais:
            corte = min(m.start() for m in finais)
            t = t[:corte]
    return t.strip()

def limpar_texto(t, lower=False, clean_edges=True):
    """Pura: NÃO escreve em variáveis globais."""
    if not isinstance(t, str):
        return ""
    t = corrigir_encoding(t)
    t = unicodedata.normalize("NFC", t)
    if clean_edges:
        t = limpar_saudacao_rodape(t)
    t = mascarar_entidades(t)
    t = re.sub(r"\b(\w+)-\n(\w+)", r"\1\2", t)
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t.lower() if lower else t


## 5 — Dataset e collator

- `TextDataset`: tokeniza com **head+tail truncation** — preserva o início (onde ficam a resposta principal) e o fim (onde costuma estar a conclusão), descartando o meio. Melhor que truncar só o fim.
- `SoftLabelCollator`: como `DataCollatorWithPadding`, mas empilha `labels` sem pad. Aceita targets suaves (vetores de probabilidade), não só classes inteiras.

In [ ]:
class TextDataset(Dataset):
    def __init__(self, texts, soft_targets, tokenizer, max_length=MAX_LENGTH):
        self.texts       = list(texts)
        self.targets     = np.asarray(soft_targets, dtype=np.float32)
        self.tokenizer   = tokenizer
        self.max_length  = max_length
        self.cls_id      = tokenizer.cls_token_id
        self.sep_id      = tokenizer.sep_token_id

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self._encode_head_tail(self.texts[idx])
        enc["labels"] = torch.tensor(self.targets[idx], dtype=torch.float32)
        return enc

    def _encode_head_tail(self, text):
        ids = self.tokenizer(text, add_special_tokens=False)["input_ids"]
        budget = self.max_length - 2
        if len(ids) > budget:
            half = budget // 2
            ids = ids[:half] + ids[-half:]
        input_ids = [self.cls_id] + list(ids) + [self.sep_id]
        return {
            "input_ids":      input_ids,
            "attention_mask": [1] * len(input_ids),
        }


class SoftLabelCollator:
    """Como DataCollatorWithPadding, mas empilha labels sem tentar padá-los."""
    def __init__(self, tokenizer):
        self.pad_collator = DataCollatorWithPadding(
            tokenizer, padding=True, return_tensors="pt"
        )
    def __call__(self, features):
        labels = [f.pop("labels") for f in features]
        batch  = self.pad_collator(features)
        batch["labels"] = torch.stack(labels, dim=0)   # (B, 3)
        return batch

## 6 — Loss e otimizador

Mudanças:
- **`focal_soft_ce_loss`**: variante de focal loss para targets suaves.
  O peso focal `(1 - p_t)^gamma` reduz a contribuição de exemplos já
  "resolvidos" e força o modelo a olhar para `c234` (a classe que sempre
  trava em F1 ~0.40). `gamma=2.0` é o valor padrão da literatura.
- **`make_scheduler`**: encapsula a escolha entre linear e cosine. Cosine
  costuma estabilizar quando `best_epoch` varia muito entre execuções.
- `build_param_groups` inalterado (LRs distintos para LoRA / classifier /
  resto continuam corretos).

In [ ]:
def soft_ce_loss(logits, soft_targets, weight=None, label_smoothing=0.0):
    if label_smoothing > 0:
        C = soft_targets.size(-1)
        soft_targets = soft_targets * (1.0 - label_smoothing) + label_smoothing / C
    log_probs = F.log_softmax(logits, dim=-1)
    per_class = -(soft_targets * log_probs)
    if weight is not None:
        per_class = per_class * weight.view(1, -1)
    return per_class.sum(dim=-1).mean()


def focal_soft_ce_loss(logits, soft_targets, weight=None,
                       gamma=2.0, label_smoothing=0.0):
    """Focal loss para targets suaves.
    Down-weighting de exemplos fáceis via (1 - p_t)^gamma.
    """
    if label_smoothing > 0:
        C = soft_targets.size(-1)
        soft_targets = soft_targets * (1.0 - label_smoothing) + label_smoothing / C
    log_probs = F.log_softmax(logits, dim=-1)
    probs = log_probs.exp()
    pt = (soft_targets * probs).sum(dim=-1, keepdim=True)   # (B,1)
    focal_w = (1.0 - pt).clamp(min=1e-6) ** gamma
    per_class = -(soft_targets * log_probs) * focal_w
    if weight is not None:
        per_class = per_class * weight.view(1, -1)
    return per_class.sum(dim=-1).mean()


def make_scheduler(optimizer, total_steps, warmup_pct, kind="linear"):
    from transformers import get_cosine_schedule_with_warmup
    nw = int(warmup_pct * total_steps)
    if kind == "cosine":
        return get_cosine_schedule_with_warmup(
            optimizer, num_warmup_steps=nw, num_training_steps=total_steps)
    return get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=nw, num_training_steps=total_steps)


def build_param_groups(model, lr_lora, lr_classifier, lr_other, weight_decay=0.01):
    lora_p, clf_p, other_p = [], [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if "lora_" in name:
            lora_p.append(p)
        elif "classifier" in name or "modules_to_save" in name:
            clf_p.append(p)
        else:
            other_p.append(p)
    groups = [{"params": lora_p, "lr": lr_lora, "weight_decay": weight_decay}]
    if clf_p:
        groups.append({"params": clf_p, "lr": lr_classifier, "weight_decay": weight_decay})
    if other_p:
        groups.append({"params": other_p, "lr": lr_other, "weight_decay": weight_decay})
    return groups

## 7 — Dados e variantes de pré-processamento

Variantes:

| Variante | Agrupamento | Target | Texto de entrada |
|---|---|---|---|
| **A** | prefix=200 | soft | limpo |
| **B** | hash exato | soft | limpo |
| **D** | prefix=200 sem conflito | hard | limpo |
| **E** | prefix=200 | soft | **cru** |

**Mudanças:**
- C removida: mesmo agrupamento de A, com alvo hard pior. Não ajuda.
- E é nova: **isola o efeito do pré-processamento**. Se E for competitiva
  com A, o pipeline de limpeza está removendo sinal útil.
- A função `montar_variante` agora aceita `text_col` para escolher entre
  `texto_limpo` e `resp_text`.

In [ ]:
dados_base = pd.read_excel(TRAIN_PATH, converters={"resp_text": str, "clarity": str})
dados_base.columns = ["resp_text", "clarity"]
dados_base = dados_base.dropna(subset=["resp_text", "clarity"]).copy()
dados_base["resp_text"] = dados_base["resp_text"].astype(str)
dados_base["clarity"]   = dados_base["clarity"].astype(str).str.strip().str.lower()
dados_base["texto_limpo"] = dados_base["resp_text"].map(limpar_texto)
mask = dados_base["texto_limpo"].str.len() >= MIN_CHARS
dados_base = dados_base[mask].reset_index(drop=True)
print(f"Total: {len(dados_base)}")
print(dados_base["clarity"].value_counts().to_dict())


def hard_from_dist(d):
    max_p = max(d.values())
    cand = [k for k, p in d.items() if p == max_p]
    for c in CLASSES:
        if c in cand:
            return c
    return cand[0]


def montar_variante(dados_df, grupo_col, usar_soft, nome, text_col="texto_limpo"):
    """text_col: 'texto_limpo' (padrão) ou 'resp_text' (variante E)."""
    cont = defaultdict(Counter)
    for g, c in zip(dados_df[grupo_col].tolist(), dados_df["clarity"].tolist()):
        cont[g][c] += 1
    mapa_dist = {g: {k: v / sum(c.values()) for k, v in c.items()}
                 for g, c in cont.items()}
    mapa_hard = {g: hard_from_dist(d) for g, d in mapa_dist.items()}

    y_soft, y_hard, y_true, grupos_out = [], [], [], []
    for _, row in dados_df.iterrows():
        g = row[grupo_col]
        d = mapa_dist[g]
        v = np.array([d.get(c, 0.0) for c in CLASSES], dtype=np.float32)
        if not usar_soft:
            vh = np.zeros(len(CLASSES), dtype=np.float32)
            vh[class_to_idx[mapa_hard[g]]] = 1.0
            v = vh
        y_soft.append(v)
        y_hard.append(mapa_hard[g])
        y_true.append(class_to_idx[row["clarity"]])
        grupos_out.append(g)

    n_conf = sum(1 for d in mapa_dist.values() if len(d) > 1)
    print(f"\n=== {nome} ===")
    print(f"  N: {len(dados_df)} | Grupos: {len(mapa_dist)} | Com conflito: {n_conf}")
    return (dados_df[text_col].tolist(),
            np.stack(y_soft), y_hard, np.array(y_true), grupos_out)


grupo_A = dados_base["texto_limpo"].map(normalizar_grupo)
grupo_B = dados_base["texto_limpo"].map(hash_grupo)

dados_A = dados_base.copy(); dados_A["grupo"] = grupo_A
dados_B = dados_base.copy(); dados_B["grupo"] = grupo_B

nunique_por_grupo = dados_A.groupby("grupo")["clarity"].nunique()
grupos_limpos = set(nunique_por_grupo[nunique_por_grupo == 1].index)
mask_limpo = dados_A["grupo"].isin(grupos_limpos)
dados_D = dados_A[mask_limpo].reset_index(drop=True)

X_A, Ys_A, Yh_A, Yt_A, G_A = montar_variante(dados_A, "grupo", True,  "A: prefix200+soft")
X_B, Ys_B, Yh_B, Yt_B, G_B = montar_variante(dados_B, "grupo", True,  "B: hash+soft")
X_D, Ys_D, Yh_D, Yt_D, G_D = montar_variante(dados_D, "grupo", False, "D: limpo+hard")
X_E, Ys_E, Yh_E, Yt_E, G_E = montar_variante(dados_A, "grupo", True,  "E: prefix200+soft+CRU",
                                              text_col="resp_text")   # NOVO

# SPLIT (fold 0 por padrão; seção 11 reusa os índices)
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
splits = list(sgkf.split(dados_A, Yh_A, groups=grupo_A))

sgkf_D = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
splits_D = list(sgkf_D.split(dados_D, dados_D["clarity"], groups=dados_D["grupo"]))


def subset(X_all, Ys_all, Yh_all, Yt_all, tr, va):
    return ([X_all[i] for i in tr], Ys_all[tr],
            [X_all[i] for i in va], Ys_all[va],
            [Yh_all[i] for i in tr], [Yh_all[i] for i in va],
            Yt_all[tr], Yt_all[va])


# VAR[var][fold] = tupla do subset. Vamos montar todos os folds de uma vez.
VAR = {}
for nome in ["A", "B", "E"]:
    Xs, Ys, Yh, Yt = {"A": (X_A, Ys_A, Yh_A, Yt_A),
                      "B": (X_B, Ys_B, Yh_B, Yt_B),
                      "E": (X_E, Ys_E, Yh_E, Yt_E)}[nome]
    VAR[nome] = {f: subset(Xs, Ys, Yh, Yt, *splits[f]) for f in FOLDS_TO_RUN}

VAR["D"] = {f: subset(X_D, Ys_D, Yh_D, Yt_D, *splits_D[f]) for f in FOLDS_TO_RUN}

Total: 19790
{'c5': 6778, 'c234': 6757, 'c1': 6255}

=== A: prefix200+soft ===
  N: 19790 | Grupos: 15823 | Com conflito: 600

=== B: hash+soft ===
  N: 19790 | Grupos: 17118 | Com conflito: 430

=== D: limpo+hard ===
  N: 15702 | Grupos: 15223 | Com conflito: 0

=== E: prefix200+soft+CRU ===
  N: 19790 | Grupos: 15823 | Com conflito: 600


## 8 — Baseline oficial reproduzido

Reproduz o `baseline.ipynb` do enunciado: TF-IDF + LogReg com **StratifiedKFold 10 folds** sobre o texto cru.

Serve como referência absoluta (o enunciado exige superar essa acurácia). O valor esperado é ~46,36%.

In [ ]:
X_oficial = dados_base["resp_text"].tolist()
Y_oficial = dados_base["clarity"].tolist()

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
folds_oficial = list(skf.split(X_oficial, Y_oficial))

# Baseline oficial CV (deve dar ~46.36%)
from sklearn.metrics import accuracy_score
accs = []
for tr, va in folds_oficial:
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(analyzer="word", ngram_range=(1,2), min_df=5,
                                  max_df=0.95, max_features=20000,
                                  sublinear_tf=True, strip_accents="unicode")),
        ("clf", LogisticRegression(C=0.5, class_weight="balanced", max_iter=150,
                                   solver="saga", tol=1e-3, random_state=42)),
    ])
    pipe.fit([X_oficial[i] for i in tr], [Y_oficial[i] for i in tr])
    accs.append(pipe.score([X_oficial[i] for i in va], [Y_oficial[i] for i in va]))
print(f"Baseline oficial CV: {sum(accs)/len(accs):.4f}")  # deve bater ~46.36%

Baseline oficial CV: 0.4617


## 8.1 — Comparação justa com o baseline oficial

O baseline oficial do enunciado é **TF-IDF + Regressão Logística** treinado sobre o **texto cru** com **rótulos originais** em validação cruzada de 10 folds. Esse é o número (~46,36%) que precisa ser superado.

Porém, o baseline calculado na seção anterior é treinado e avaliado **no mesmo split do BERT+LoRA** (fold 0), o que garante um delta honesto, mas não é o número do enunciado. Para não misturar as duas coisas, esta seção calcula **três baselines**:

| Baseline | Texto | Rótulo | Split | Para que serve |
|---|---|---|---|---|
| **`baseline_oficial`** | cru | original | 10 folds próprios (`StratifiedKFold`) | Reproduzir o número do enunciado (~46,36%) como referência absoluta. |
| **`baseline_ABC_split`** | cru | original | mesmo fold 0 do BERT (A/B/C) | Delta justo para A/B/C: mesma partição de treino/validação, mesmo conjunto de instâncias. |
| **`baseline_D_split`** | cru | original | mesmo fold 0 do BERT (D) | Delta justo para D (subset filtrado, split próprio). |

**Por quê os três**: o `delta_oficial` responde à exigência do enunciado ("resultado inferior ao baseline oficial"), mas compara splits diferentes — útil como referência, não como comparação limpa. O `delta_split` é a comparação correta: mesmo conjunto de validação, mesma instância, mesmo rótulo original. É esse número que deve ser destacado no relatório e no slide.

**Ponto importante**: o TF-IDF do baseline sempre usa **texto cru** e **rótulo original**, mesmo quando o BERT+LoRA usa **texto pré-processado** e **alvos suavizados** (soft labels de grupo). Essa assimetria é intencional — o baseline do enunciado é definido assim, e o denoising (soft labels no treino) é a contribuição do nosso modelo, não do baseline.

In [ ]:
def make_tfidf_logreg():
    return Pipeline([
        ("tfidf", TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=5,
                                  max_df=0.95, max_features=20000,
                                  sublinear_tf=True, strip_accents="unicode")),
        ("clf", LogisticRegression(C=0.5, class_weight="balanced", max_iter=150,
                                   solver="saga", tol=1e-3, random_state=42)),
    ])

# --- 1) Baseline OFICIAL: 10-fold CV sobre texto cru + rótulo original ---
X_cru   = dados_base["resp_text"].tolist()
Y_orig  = dados_base["clarity"].tolist()

# Baseline justo usa o MESMO fold 0 que as variantes
tr_idx_A, va_idx_A = splits[0]
tr_idx_D, va_idx_D = splits_D[0]

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
accs_oficial = []
for tr, va in skf.split(X_cru, Y_orig):
    pipe = make_tfidf_logreg()
    pipe.fit([X_cru[i] for i in tr], [Y_orig[i] for i in tr])
    accs_oficial.append(pipe.score([X_cru[i] for i in va], [Y_orig[i] for i in va]))
baseline_oficial = float(np.mean(accs_oficial))
print(f"[Oficial 10-fold CV] acc = {baseline_oficial:.4f}  (esperado ~0.4636)")

# --- 2) Baseline no MESMO split do BERT (fold 0 de A/B/C) ---
# tr_idx_A / va_idx_A são posições em dados_A, que preserva a ordem de dados_base.
X_tr_raw_ABC = [dados_base["resp_text"].iloc[i] for i in tr_idx_A]
X_va_raw_ABC = [dados_base["resp_text"].iloc[i] for i in va_idx_A]
Y_tr_raw_ABC = [dados_base["clarity"].iloc[i] for i in tr_idx_A]   # rótulo ORIGINAL
Y_va_raw_ABC = [dados_base["clarity"].iloc[i] for i in va_idx_A]   # rótulo ORIGINAL

pipe_ABC = make_tfidf_logreg()
pipe_ABC.fit(X_tr_raw_ABC, Y_tr_raw_ABC)
baseline_ABC_split = float(pipe_ABC.score(X_va_raw_ABC, Y_va_raw_ABC))
print(f"[Split A/B/C, texto cru + rótulo original] acc = {baseline_ABC_split:.4f}")

# --- 3) Baseline no MESMO split do BERT (fold 0 de D) ---
# tr_idx_D / va_idx_D são posições em dados_D, que é um subset de dados_A.
# Precisamos mapear de volta para dados_base pelo índice original.
idx_original_D = np.where(mask_limpo.to_numpy())[0]
tr_orig_D = idx_original_D[tr_idx_D]
va_orig_D = idx_original_D[va_idx_D]

X_tr_raw_D = [dados_base["resp_text"].iloc[i] for i in tr_orig_D]
X_va_raw_D = [dados_base["resp_text"].iloc[i] for i in va_orig_D]
Y_tr_raw_D = [dados_base["clarity"].iloc[i] for i in tr_orig_D]
Y_va_raw_D = [dados_base["clarity"].iloc[i] for i in va_orig_D]

pipe_D_raw = make_tfidf_logreg()
pipe_D_raw.fit(X_tr_raw_D, Y_tr_raw_D)
baseline_D_split = float(pipe_D_raw.score(X_va_raw_D, Y_va_raw_D))
print(f"[Split D, texto cru + rótulo original]   acc = {baseline_D_split:.4f}")

print("\nResumo:")
print(f"  baseline_oficial     = {baseline_oficial:.4f}")
print(f"  baseline_ABC_split   = {baseline_ABC_split:.4f}")
print(f"  baseline_D_split     = {baseline_D_split:.4f}")

[Oficial 10-fold CV] acc = 0.4617  (esperado ~0.4636)
[Split A/B/C, texto cru + rótulo original] acc = 0.4685
[Split D, texto cru + rótulo original]   acc = 0.4468

Resumo:
  baseline_oficial     = 0.4617
  baseline_ABC_split   = 0.4685
  baseline_D_split     = 0.4468


## 9 — Função de treino (1 fold, 1 seed)

Mudanças:
- **`seed` como parâmetro**: permite chamar a mesma função com seeds
  diferentes para o ensemble.
- **`focal_gamma` e `scheduler` lidos de `cfg`**: a função escolhe entre
  `soft_ce_loss` e `focal_soft_ce_loss`, e entre linear e cosine.
- **Retorna `val_logits` e `val_probs`**: o ensemble precisa das
  probabilidades (ou logits) para combinar modelos.

In [ ]:
def treinar_fold(cfg, train_texts, train_soft, val_texts, val_soft,
                 val_true, log, seed=None):
    if seed is None:
        seed = SEED
    set_seed(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    le = LabelEncoder(); le.fit(CLASSES)
    num_labels = len(le.classes_)

    tokenizer = AutoTokenizer.from_pretrained(cfg["model"])
    base = AutoModelForSequenceClassification.from_pretrained(
        cfg["model"], num_labels=num_labels)
    lora_cfg = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=cfg["lora_r"], lora_alpha=cfg["lora_alpha"],
        lora_dropout=cfg["lora_dropout"],
        target_modules=cfg["target_modules"],
        modules_to_save=["classifier"],
    )
    model = get_peft_model(base, lora_cfg).to(device)

    train_ds = TextDataset(train_texts, train_soft, tokenizer, cfg["max_length"])
    val_ds   = TextDataset(val_texts,   val_soft,   tokenizer, cfg["max_length"])
    collator = SoftLabelCollator(tokenizer)

    train_loader = DataLoader(train_ds, batch_size=cfg["batch"], shuffle=True,
                              collate_fn=collator, num_workers=0)
    val_loader   = DataLoader(val_ds, batch_size=cfg["batch"] * 2,
                              collate_fn=collator, num_workers=0)

    y_hard_train = np.argmax(np.asarray(train_soft), axis=1)
    pesos = compute_class_weight("balanced", classes=np.arange(num_labels), y=y_hard_train)
    pesos_t = torch.tensor(pesos, dtype=torch.float, device=device)

    gamma = float(cfg.get("focal_gamma", 0.0))
    def loss_fn(logits, soft):
        if gamma > 0:
            return focal_soft_ce_loss(logits, soft, weight=pesos_t,
                                      gamma=gamma,
                                      label_smoothing=cfg["label_smoothing"])
        return soft_ce_loss(logits, soft, weight=pesos_t,
                            label_smoothing=cfg["label_smoothing"])

    optimizer = AdamW(
        build_param_groups(model, cfg["lr_lora"], cfg["lr_clf"], cfg["lr_lora"]),
        weight_decay=0.01)
    grad_accum = cfg["grad_accum"]
    steps_per_epoch = (len(train_loader) + grad_accum - 1) // grad_accum
    total_steps = steps_per_epoch * cfg["epochs"]
    scheduler = make_scheduler(optimizer, total_steps, WARMUP_PCT,
                               kind=cfg.get("scheduler", "linear"))

    best_f1, best_acc, best_epoch = -1.0, -1.0, -1
    best_state = None
    t0 = time.time()

    for epoch in range(cfg["epochs"]):
        model.train(); optimizer.zero_grad()
        epoch_losses = []
        for step, batch in enumerate(train_loader):
            batch = {k: v.to(device) for k, v in batch.items()}
            soft = batch.pop("labels")
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                logits = model(**batch).logits
                loss = loss_fn(logits.float(), soft) / grad_accum
            loss.backward()
            epoch_losses.append(loss.item() * grad_accum)
            if (step + 1) % grad_accum == 0 or (step + 1) == len(train_loader):
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()

        model.eval()
        preds = []
        with torch.no_grad():
            for batch in val_loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                batch.pop("labels")
                logits = model(**batch).logits.float()
                preds.extend(logits.argmax(dim=1).cpu().numpy())

        reais = list(val_true)
        f1 = f1_score(reais, preds, average="macro")
        acc = accuracy_score(reais, preds)
        mcc = matthews_corrcoef(reais, preds)
        lm = float(np.mean(epoch_losses))
        log(f"  [seed={seed}] Epoch {epoch+1}/{cfg['epochs']} | loss={lm:.4f} | "
            f"F1={f1:.4f} | Acc={acc:.4f} | MCC={mcc:.4f}")

        if (f1 > best_f1) or (f1 == best_f1 and acc > best_acc):
            best_f1, best_acc, best_epoch = f1, acc, epoch + 1
            best_state = {k: v.detach().cpu().clone()
                          for k, v in model.state_dict().items()
                          if "lora_" in k or "classifier" in k or "modules_to_save" in k}

    # restaura melhor estado e recomputa probs
    if best_state is not None:
        model.load_state_dict(best_state, strict=False)
    model.eval()

    logits_all, preds_final = [], []
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            batch.pop("labels")
            logits = model(**batch).logits.float()
            logits_all.append(logits.cpu().numpy())
            preds_final.extend(logits.argmax(dim=1).cpu().numpy())

    val_logits = np.concatenate(logits_all, axis=0)
    val_probs = torch.softmax(torch.tensor(val_logits), dim=-1).numpy()

    elapsed = time.time() - t0
    del model, base, optimizer, scheduler, train_loader, val_loader
    del train_ds, val_ds, collator
    gc.collect(); torch.cuda.empty_cache()

    return {"best_f1": best_f1, "best_acc": best_acc, "best_epoch": best_epoch,
            "elapsed_min": elapsed / 60,
            "val_preds": np.array(preds_final),
            "val_logits": val_logits,
            "val_probs": val_probs}

## 10 — Ensemble multi-seed

**Por quê:** o `best_epoch` oscilava entre 2 e 5 nos experimentos anteriores.
Isso indica que cada execução converge para um ponto diferente do espaço de
parâmetros. Média de probabilidades de 3 seeds reduz essa variância e é o
ganho mais barato disponível.

**Como:** treina 3 modelos com seeds diferentes, faz a média das
probabilidades no conjunto de validação (soft-voting) e reporta as métricas
do ensemble. Compara com a média individual das seeds — se o ensemble for
melhor que a média, houve diversidade útil.

In [ ]:
def treinar_fold_ensemble(cfg, train_texts, train_soft, val_texts, val_soft,
                          val_true, seeds, log):
    probs_list, metrics_list = [], []
    for seed in seeds:
        res = treinar_fold(cfg, train_texts, train_soft,
                           val_texts, val_soft, val_true, log, seed=seed)
        probs_list.append(res["val_probs"])
        metrics_list.append({
            "seed": seed,
            "acc": res["best_acc"],
            "f1": res["best_f1"],
            "epoch": res["best_epoch"],
            "t_min": res["elapsed_min"],
        })
        log(f"    seed={seed} | acc={res['best_acc']:.4f} | "
            f"f1={res['best_f1']:.4f} | ep={res['best_epoch']}")

    mean_probs = np.mean(probs_list, axis=0)
    preds = mean_probs.argmax(axis=1)
    reais = list(val_true)

    f1_ens = f1_score(reais, preds, average="macro")
    acc_ens = accuracy_score(reais, preds)
    mcc_ens = matthews_corrcoef(reais, preds)

    acc_media = float(np.mean([m["acc"] for m in metrics_list]))
    f1_media  = float(np.mean([m["f1"]  for m in metrics_list]))
    std_acc   = float(np.std([m["acc"] for m in metrics_list]))

    log(f"    [ensemble] acc={acc_ens:.4f} | f1={f1_ens:.4f} | mcc={mcc_ens:.4f}")
    log(f"    [seeds]    acc_media={acc_media:.4f} ± {std_acc:.4f} | f1_media={f1_media:.4f}")

    return {
        "ensemble_acc": acc_ens,
        "ensemble_f1": f1_ens,
        "ensemble_mcc": mcc_ens,
        "seed_acc_mean": acc_media,
        "seed_acc_std": std_acc,
        "seed_f1_mean": f1_media,
        "seeds_detail": metrics_list,
        "val_probs": mean_probs,
    }

## 11 — Execução dos experimentos

Loop sobre:
- **Variantes** (A, B, D, E)
- **Folds** (0, 1, 2)
- **Seeds** (42, 123, 2024) via `treinar_fold_ensemble`

Para cada (variante, fold), reporta acurácia/F1/MCC do ensemble, além da
média ± desvio individual das seeds. Armazena tudo em CSV para a tabela
final.

**Por que rodar 3 folds × 3 seeds × 4 variantes:** com 2237 amostras por
fold de validação, 1 ponto percentual ≈ 22 amostras. Precisamos saber se
a diferença entre variantes é sistemática ou ruído. Só com std entre folds
conseguimos responder isso.

In [ ]:
OUT_DIR = DRIVE_ROOT / "resultados" / "v9_final"
OUT_DIR.mkdir(parents=True, exist_ok=True)

resultados = []   # lista de dicts: variante, fold, métricas

for nome in VARIANTES:
    for fold in FOLDS_TO_RUN:
        X_tr, Y_tr_s, X_va, Y_va_s, _, _, _, Yt_va = VAR[nome][fold]
        tag = f"var{nome}_fold{fold}"
        log = Logger(OUT_DIR / f"{tag}.txt")
        log("=" * 72)
        log(f"VARIANTE {nome} | FOLD {fold} | "
            f"Treino: {len(X_tr)} | Val: {len(X_va)}")
        log(f"cfg: focal_gamma={CFG_TREINO['focal_gamma']} | "
            f"scheduler={CFG_TREINO['scheduler']} | "
            f"lr_clf={CFG_TREINO['lr_clf']} | epochs={CFG_TREINO['epochs']}")
        log("=" * 72)

        base = baseline_D_split if nome == "D" else baseline_ABC_split

        res = treinar_fold_ensemble(
            CFG_TREINO, X_tr, Y_tr_s, X_va, Y_va_s, Yt_va,
            seeds=SEEDS_ENSEMBLE, log=log)

        delta_split   = res["ensemble_acc"] - base
        delta_oficial = res["ensemble_acc"] - baseline_oficial

        log(f">>> {nome} fold {fold}: "
            f"ens_acc={res['ensemble_acc']:.4f} | ens_f1={res['ensemble_f1']:.4f} | "
            f"seed_acc={res['seed_acc_mean']:.4f}±{res['seed_acc_std']:.4f} | "
            f"delta_split={delta_split:+.4f} | delta_oficial={delta_oficial:+.4f}")
        log.close()

        resultados.append({
            "variante": nome,
            "fold": fold,
            "base": base,
            "ens_acc": res["ensemble_acc"],
            "ens_f1": res["ensemble_f1"],
            "ens_mcc": res["ensemble_mcc"],
            "seed_acc_mean": res["seed_acc_mean"],
            "seed_acc_std":  res["seed_acc_std"],
            "seed_f1_mean":  res["seed_f1_mean"],
            "delta_split":   delta_split,
            "delta_oficial": delta_oficial,
            "epoch_mean": float(np.mean([m["epoch"] for m in res["seeds_detail"]])),
        })
        gc.collect(); torch.cuda.empty_cache()

df_res = pd.DataFrame(resultados)
df_res.to_csv(OUT_DIR / "resultados_v9.csv", index=False)

# Resumo agregado por variante (média entre folds)
df_agg = df_res.groupby("variante").agg(
    ens_acc_mean=("ens_acc", "mean"),
    ens_acc_std =("ens_acc", "std"),
    ens_f1_mean =("ens_f1", "mean"),
    seed_acc_mean=("seed_acc_mean", "mean"),
    seed_acc_std_mean=("seed_acc_std", "mean"),
    delta_split_mean=("delta_split", "mean"),
    delta_oficial_mean=("delta_oficial", "mean"),
).sort_values("ens_acc_mean", ascending=False)

print("\n" + "=" * 90)
print("RESULTADO AGREGADO (média entre folds)")
print(f"  baseline_oficial   = {baseline_oficial:.4f}")
print(f"  baseline_ABC_split = {baseline_ABC_split:.4f}")
print(f"  baseline_D_split   = {baseline_D_split:.4f}")
print("=" * 90)
print(df_agg.to_string())
df_agg.to_csv(OUT_DIR / "resumo_v9.csv")

VARIANTE A | FOLD 0 | Treino: 17553 | Val: 2237
cfg: focal_gamma=2.0 | scheduler=cosine | lr_clf=0.0005 | epochs=6


config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

...

[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from th

  [seed=123] Epoch 1/6 | loss=0.4818 | F1=0.4072 | Acc=0.4217 | MCC=0.1448
  [seed=123] Epoch 2/6 | loss=0.4596 | F1=0.4294 | Acc=0.4533 | MCC=0.2103
  [seed=123] Epoch 3/6 | loss=0.4460 | F1=0.4826 | Acc=0.4807 | MCC=0.2202
  [seed=123] Epoch 4/6 | loss=0.4323 | F1=0.4622 | Acc=0.4759 | MCC=0.2169
  [seed=123] Epoch 5/6 | loss=0.4222 | F1=0.4707 | Acc=0.4732 | MCC=0.2068
  [seed=123] Epoch 6/6 | loss=0.4154 | F1=0.4686 | Acc=0.4732 | MCC=0.2072
    seed=123 | acc=0.4807 | f1=0.4826 | ep=3


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from th

  [seed=2024] Epoch 1/6 | loss=0.4811 | F1=0.3907 | Acc=0.4437 | MCC=0.1955
  [seed=2024] Epoch 2/6 | loss=0.4600 | F1=0.4618 | Acc=0.4689 | MCC=0.1996
  [seed=2024] Epoch 3/6 | loss=0.4431 | F1=0.4741 | Acc=0.4732 | MCC=0.2067
  [seed=2024] Epoch 4/6 | loss=0.4292 | F1=0.4594 | Acc=0.4641 | MCC=0.1915
  [seed=2024] Epoch 5/6 | loss=0.4185 | F1=0.4649 | Acc=0.4683 | MCC=0.1982
  [seed=2024] Epoch 6/6 | loss=0.4139 | F1=0.4598 | Acc=0.4657 | MCC=0.1958
    seed=2024 | acc=0.4732 | f1=0.4741 | ep=3
    [ensemble] acc=0.4759 | f1=0.4778 | mcc=0.2118
    [seeds]    acc_media=0.4771 ± 0.0031 | f1_media=0.4785
>>> E fold 2: ens_acc=0.4759 | ens_f1=0.4778 | seed_acc=0.4771±0.0031 | delta_split=+0.0074 | delta_oficial=+0.0142

RESULTADO AGREGADO (média entre folds)
  baseline_oficial   = 0.4617
  baseline_ABC_split = 0.4685
  baseline_D_split   = 0.4468
          ens_acc_mean  ens_acc_std  ens_f1_mean  seed_acc_mean  seed_acc_std_mean  delta_split_mean  delta_oficial_mean
variante             

## 12 — Treino final + predição do teste

Muda em relação ao v8(11):
- **Usa o ensemble** (3 seeds) na predição do teste, com média de probabilidades.
- **Usa focal loss + cosine**, coerente com o treino dos folds.
- **`epochs_fixo`** = média arredondada dos `best_epoch` entre folds e seeds.

Se o stacking (seção 12) vencer nos folds, essa seção também pode ser
adaptada para treinar um stacker final — mas exige manter o TF-IDF
serializado para uso na inferência.

In [ ]:
FULL_OUT = DRIVE_ROOT / "resultados" / "v9_full"
FULL_OUT.mkdir(parents=True, exist_ok=True)
log = Logger(FULL_OUT / "log.txt")
log("=" * 72)
log(f"Treino final (ensemble {len(SEEDS_ENSEMBLE)} seeds) — "
    f"{datetime.now():%Y-%m-%d %H:%M:%S}")
log("=" * 72)

# Épocas fixas = média dos best_epoch observados
try:
    ep_fixo = int(round(df_res["epoch_mean"].mean()))
    ep_fixo = max(2, min(ep_fixo, CFG_TREINO["epochs"]))
except Exception:
    ep_fixo = CFG_TREINO["epochs"]
ep_fixo = max(2, min(ep_fixo, CFG_TREINO["epochs"]))
log(f"Épocas fixas no treino final: {ep_fixo}")

# Reaproveita a melhor variante do resumo
melhor_var = df_agg.index[0]
log(f"Melhor variante pelo ens_acc: {melhor_var}")

# Reconstrói o dataset completo da melhor variante
Xs_full, Ys_full, _, _, _ = {
    "A": (X_A, Ys_A, None, None, None),
    "B": (X_B, Ys_B, None, None, None),
    "D": (X_D, Ys_D, None, None, None),
    "E": (X_E, Ys_E, None, None, None),
}[melhor_var]

test_raw = pd.read_excel(TEST_PATH, converters={"resp_text": str})["resp_text"].tolist()
if melhor_var == "E":
    X_test = test_raw                                # coerente com o treino
else:
    X_test = [limpar_texto(t) for t in test_raw]
log(f"Treino completo: {len(Xs_full)} | Teste: {len(X_test)}")

# Treina N seeds no full e agrega
test_probs_all = []
for seed in SEEDS_ENSEMBLE:
    set_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    le = LabelEncoder(); le.fit(CLASSES)

    tokenizer = AutoTokenizer.from_pretrained(CFG_TREINO["model"])
    base = AutoModelForSequenceClassification.from_pretrained(
        CFG_TREINO["model"], num_labels=len(le.classes_))
    lora_cfg = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=CFG_TREINO["lora_r"], lora_alpha=CFG_TREINO["lora_alpha"],
        lora_dropout=CFG_TREINO["lora_dropout"],
        target_modules=CFG_TREINO["target_modules"],
        modules_to_save=["classifier"])
    model = get_peft_model(base, lora_cfg).to(device)

    train_ds = TextDataset(Xs_full, Ys_full, tokenizer, CFG_TREINO["max_length"])
    collator = SoftLabelCollator(tokenizer)
    train_loader = DataLoader(train_ds, batch_size=CFG_TREINO["batch"],
                              shuffle=True, collate_fn=collator, num_workers=0)

    y_hard_train = np.argmax(np.asarray(Ys_full), axis=1)
    pesos = compute_class_weight("balanced", classes=np.arange(len(CLASSES)),
                                 y=y_hard_train)
    pesos_t = torch.tensor(pesos, dtype=torch.float, device=device)

    def loss_fn(logits, soft):
        return focal_soft_ce_loss(logits, soft, weight=pesos_t,
                                  gamma=CFG_TREINO["focal_gamma"],
                                  label_smoothing=CFG_TREINO["label_smoothing"])

    optimizer = AdamW(build_param_groups(
        model, CFG_TREINO["lr_lora"], CFG_TREINO["lr_clf"], CFG_TREINO["lr_lora"]),
        weight_decay=0.01)
    steps_per_epoch = (len(train_loader) + CFG_TREINO["grad_accum"] - 1) // CFG_TREINO["grad_accum"]
    total_steps = steps_per_epoch * ep_fixo
    scheduler = make_scheduler(optimizer, total_steps, WARMUP_PCT,
                               kind=CFG_TREINO["scheduler"])

    model.train(); optimizer.zero_grad()
    for epoch in range(ep_fixo):
        losses = []
        for step, batch in enumerate(train_loader):
            batch = {k: v.to(device) for k, v in batch.items()}
            soft = batch.pop("labels")
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                logits = model(**batch).logits
                loss = loss_fn(logits.float(), soft) / CFG_TREINO["grad_accum"]
            loss.backward()
            losses.append(loss.item() * CFG_TREINO["grad_accum"])
            if (step + 1) % CFG_TREINO["grad_accum"] == 0 or (step + 1) == len(train_loader):
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()
        log(f"  [seed={seed}] ep {epoch+1}/{ep_fixo} | loss={np.mean(losses):.4f}")

    # predição do teste
    test_ds = TextDataset(X_test, np.tile(np.array([1/3,1/3,1/3], dtype=np.float32),
                                          (len(X_test), 1)),
                          tokenizer, CFG_TREINO["max_length"])
    test_loader = DataLoader(test_ds, batch_size=CFG_TREINO["batch"]*2,
                             collate_fn=collator)
    model.eval()
    probs = []
    with torch.no_grad():
        for batch in test_loader:
            batch = {k: v.to(device) for k, v in batch.items() if k != "labels"}
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                logits = model(**batch).logits
            probs.append(torch.softmax(logits.float(), dim=-1).cpu().numpy())
    test_probs_all.append(np.concatenate(probs, axis=0))

    del model, base, optimizer, scheduler, train_loader, test_loader
    gc.collect(); torch.cuda.empty_cache()

# média dos seeds
test_probs = np.mean(test_probs_all, axis=0)
y_pred = test_probs.argmax(axis=1)
y_pred_labels = le.inverse_transform(y_pred)

teste_rotulado = pd.DataFrame({"resp_text": X_test, "clarity": y_pred_labels})
out_path = FULL_OUT / "teste_rotulado.xlsx"
teste_rotulado.to_excel(out_path, index=False)
log(f"✓ Salvo: {out_path}")
log(f"Distribuição das predições: {teste_rotulado['clarity'].value_counts().to_dict()}")
log.close()
print("\n✓ Pronto: teste_rotulado.xlsx")